# 1. Data Preprocessing and Exploratory Data Analysis
This step is quality control and handles missing values, outliers and data types by cleaning the dataset. Then the distribution of clinical variables is explored. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, shapiro
import sys
import missingno as msno
import warnings
import os

warnings.filterwarnings('ignore')

#set global plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13, 
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
})
sns.set_style("whitegrid")


In [3]:
excel_path = 'polonen.xlsx'
clinical = 'ST1_Clinical_Data'
output_dir = 'eda_outputs'
os.makedirs(output_dir, exist_ok=True)

patient_id = 'USI'
id_cols = [
    "USI", "Sort.For.Germline.DNA", "WGS.Germline.DNA.Sample.ID", 
    "WGS.Tumor.DNA.Sample.ID", "WES.Tumor.DNA.Sample.ID", 
    "RNAseq.Tumor.RNA.Sample.ID", "Relapse.WES.Tumor.DNA",
    "Relapse.RNAseq.Tumor.RNA", "Relapse.WGS.Tumor.DNA",
    "RNAseq.Tumor.Backup.Samples", "WES.Tumor.Backup.Samples",
    "WGS.Tumor.Backup.Samples", "combio_name_D", "combio_name_G",
    "combio_name_R", "combio_name_D.alias", "sample_sjuid", "subject_sjuid",
]

diagnosis_cols = [
    "Diagnosis.Text", "Diagnosis.Mondo", "Diagnosis.ICD-10", "Diagnosis.NCIt", 
    "Clinical.Trial.Protocol.Name",
]

continuous_vars = [
    "Age.at.Diagnosis.in.Years",
    "Age.at.Diagnosis.in.Days",
    "WBC.at.Diagnosis",
    "Percent.Blasts.Tumor.Sample.Diagnostic",
    "Germline.with.Tumor.Contamination.%.Blasts",
]

binary_vars = [
    "In.TARGET.Cohort. (n=.264). RNASeq. and. WES", #Yes/No or 1/0
    "Sex", #M/F
    "ETP.STATUS", #ETP/non-ETP (Early T-cell Precursor)
]

categorical_vars = [
    "Missing.Information", #flags for data gaps
    "CNS.Status", # CNS1/CNS2/CNS3
    "Cause.of.Death", #cause categories
    "Event.Type", #type of event (relapse, death, etc.)
    "Day.29.MRD", #MRD: Positive/Negative/Unknown
    "End.of.Consolidation.MRD", #same scale
    "Tumor.Specimen.Type", 
    "Germline.Specimen.Type", 
    "Relapse.Specimen.Type", 
]

survival_endpoints = {
    "OS": {"time": "OS", "event": "OS.status"},
    "EFS": {"time": "EFS", "event": "EFS.status"},
    "DFS": {"time": "DFS", "event": "DFS.status"}
}

primary_time = "OS"
primary_event = "OS.status"

efs_days_col = "Event-free.Survival.in.Days"

### TARGET COHORT - PREPROCESSING & EDA PIPELINE

#### STEP 0 - Load Clinical Sheet

In [4]:
xl = pd.ExcelFile(excel_path)
print(f" All sheets in file: {xl.sheet_names}\n")

 All sheets in file: ['Summary', 'ST1_Clinical_Data', 'ST2_Cohort Composition', 'ST3_Sample_Annotations', 'ST4_UMAP_Coordinates', 'ST5_UMAP_features', 'ST6_Driver.subtype.freq', 'ST7_IP MRD Panel', 'ST8_IP Data', 'ST9_Alterations.annotations', 'ST10_Alterations.genes', 'ST11_Alterations.variants', 'ST12_Alterations.Pathways', 'ST13_Pathway.annotations', 'ST14_Alterations.SNVIndel', 'ST15_Alterations.Pindel', 'ST16_Alterations.SV', 'ST17_Alterations.SV.All', 'ST18_Alterations.RNASV', 'ST19_Alterations.CNV', 'ST20_Stats.GRIN2.summarized', 'ST21.Stats.GRIN2.coding', 'ST22_Stats.GRIN2.ncoding', 'ST23_Stats.genes.freq', 'ST24_Stats.Monoallelic.exp', 'ST25_Stats.Subtypes', 'ST26_RAG_breakpoints', 'ST27_stats.RAG_breakpoints', 'ST28_Stats.TLX3', 'ST29_Stats.NKX2-1', 'ST30_Stats.TAL1', 'ST31_Stats.TAL1 subgroups', 'ST32_Stats.ETP-like', 'ST33_Stats.Subtypes.Up', 'ST34_Stats.Subtypes.Down', 'ST35_Stats.MED12ko.LOUCY', 'ST36_Stats.Univariable.Driver', 'ST37_Stats.Univariable.ETP', 'ST38_Stats.Un

In [5]:
df_raw = pd.read_excel(excel_path, sheet_name = clinical)
print(f" Clinical sheet shape : {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f" Expected columns : 47 | Found: {df_raw.shape[1]}")

 Clinical sheet shape : 1335 rows x 51 columns
 Expected columns : 47 | Found: 51


In [6]:
# show all 47 column names with dtype and % missing side by side
snapshot = pd.DataFrame({
    "column" : df_raw.columns, 
    "dtype" : df_raw.dtypes.values,
    "n_unique" : [df_raw[c].nunique() for c in df_raw.columns],
    "pct_missing": [round(df_raw[c].isna().mean()*100, 1) for c in df_raw.columns],
    "example_val": [str(df_raw[c].dropna().iloc[0]) if df_raw[c].notna().any() 
                    else "ALL MISSING" for c in df_raw.columns],
})
snapshot.to_csv(f"{output_dir}/00_column_snapshot.csv", index=False)
print(f"\n Column snapshot saved -> {output_dir}/00_column_snapshot.csv")
print("\n Snapshot preview:")
print(snapshot.to_string(index=False))


 Column snapshot saved -> eda_outputs/00_column_snapshot.csv

 Snapshot preview:
                                    column   dtype  n_unique  pct_missing                               example_val
                                       USI  object      1335          0.0                                    PAWDLY
                     Sort.For.Germline.DNA  object         1         98.2                                      Sort
  In.TARGET.Cohort.(n=.264).RNASeq.and.WES  object         1         80.7                                       yes
                       Missing.Information  object         3         97.6                                       Yes
                WGS.Germline.DNA.Sample.ID  object      1321          0.0                             5962-DTT-1207
                   WGS.Tumor.DNA.Sample.ID  object      1333          0.0                             6159-DTT-1614
                   WES.Tumor.DNA.Sample.ID  object      1079          0.0                             6222

#### STEP 1: SEPARATE ID / TRACKING COLUMNS FROM ANALYSIS VARIABLES

Sample-tracking IDs (WGS IDs, backup sample IDs, combio names) are string labels for linking to genomic files; they add zero statistical signal and will break any numeric pipeline.

In [8]:
id_lookup = df_raw[[c for c in id_cols if c in df_raw.columns]].copy()
id_lookup.to_csv(f"{output_dir}/01_sample_id_lookup.csv", index = False)
print(f" Sample ID lookup saved -> {output_dir}/01_sample_id_lookup.csv")
print(f" ID/tracking columns isolated: {[c for c in id_cols if c in df_raw.columns]}")

 Sample ID lookup saved -> eda_outputs/01_sample_id_lookup.csv
 ID/tracking columns isolated: ['USI', 'Sort.For.Germline.DNA', 'WGS.Germline.DNA.Sample.ID', 'WGS.Tumor.DNA.Sample.ID', 'WES.Tumor.DNA.Sample.ID', 'RNAseq.Tumor.RNA.Sample.ID', 'Relapse.WES.Tumor.DNA', 'Relapse.RNAseq.Tumor.RNA', 'Relapse.WGS.Tumor.DNA', 'WES.Tumor.Backup.Samples', 'WGS.Tumor.Backup.Samples', 'combio_name_D', 'combio_name_G', 'combio_name_R', 'combio_name_D.alias', 'sample_sjuid', 'subject_sjuid']


In [9]:
#working dataframe: clinical + survival columns only 
keep_cols = (continuous_vars + binary_vars + categorical_vars + 
             diagnosis_cols + [efs_days_col] + 
             [v for ep in survival_endpoints.values()
              for v in [ep['time'], ep['event']]] + 
             [patient_id])
keep_cols = [c for c in keep_cols if c in df_raw.columns]

df = df_raw[keep_cols].copy()
print(f"\n Working DataFrame shape: {df.shape} ({df.shape[1]} clinical variables)")


 Working DataFrame shape: (1335, 29) (29 clinical variables)


In [10]:
#check for duplicate patients
dupes = df.duplicated(subset=patient_id).sum()
if dupes > 0:
    print(f"\n WARNING: {dupes} duplicate USI rows detected - keeping first occurence.")
    df = df.drop_duplicates(subset=patient_id, keep='first')
else: 
    print(f" No duplicate USI values.")

print(f" Unique patients (USI): {df[patient_id].nunique()}")

 No duplicate USI values.
 Unique patients (USI): 1335


#### STEP 2: MISSING DATA AUDIT

In TARGET/paediatric oncology datasets, missingness is often clinically meaningful: 
- missing MRD = test not performed (protocol era, risk group)
- missing Cause of Death = patient is alive (censored) → not truly missing
- missing relapse specimens = no relapse occurred

In [12]:
missing = pd.DataFrame({
    "n_missing" : df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
    "n_present": df.notna().sum(),
    "dtype" : df.dtypes,
}).sort_values("pct_missing", ascending=False)

missing.to_csv(f"{output_dir}/02_missing_audit.csv")
print(missing.to_string())

                                            n_missing  pct_missing  n_present    dtype
Germline.with.Tumor.Contamination.%.Blasts       1327         99.4          8  float64
Missing.Information                              1303         97.6         32   object
Cause.of.Death                                   1194         89.4        141   object
End.of.Consolidation.MRD                         1185         88.8        150  float64
Event-free.Survival.in.Days                      1115         83.5        220  float64
Event.Type                                       1115         83.5        220   object
DFS.status                                        320         24.0       1015  float64
DFS                                               320         24.0       1015  float64
Percent.Blasts.Tumor.Sample.Diagnostic             31          2.3       1304  float64
ETP.STATUS                                         25          1.9       1310   object
Day.29.MRD                                 

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
msno.matrix(df, ax=axes[0], sparkline=False, color=(0.18, 0.42, 0.68),
            fontsize=7)
axes[0].set_title("Missing Data Matrix\n(white stripe = missing)", fontsize=11)
msno.bar(df, ax=axes[1], color=(0.18, 0.42, 0.68), fontsize=7)
axes[1].set_title("Completeness per Variable", fontsize=11)
plt.suptitle("TARGET Cohort - Missing Data Overview", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{output_dir}/02_missing_data.png", bbox_inches="tight")
plt.close()
print(f"\n Missing data plot saved -> {output_dir}/02_missing_data.png")


 Missing data plot saved -> eda_outputs/02_missing_data.png


Clinically-informed missing flags important. Cause of Death is only meaningful in patients who died. Treating it as "missing" for alive patients is wrong. Create a binary flag instead of imputing. 

In [18]:
#flag: patient died (has Cause of Death) vs. alive / unknown
if "Cause.of.Death" in df.columns:
    df["Death_recorded"] = df["Cause.of.Death"].notna().astype(int)
    print("Death_recorded flag (1=death recorded, 0=no death/unknown)")

#flag: relapse specimens exist -> patient relapsed
relapse_specimen_col = "Relapse.Specimen.Type"
if relapse_specimen_col in df.columns:
    df["Had_Relapse_Specimen"] = df[relapse_specimen_col].notna().astype(int)
    print("Had_Relapse_Specimen flag (1=relapse specimen available)")

#flag: MRD tested at Day 29
if "Day.29.MRD" in df.columns:
    df["Day29_MRD_tested"] = df["Day.29.MRD"].notna().astype(int)
    print("Day29_MRD_tested flag (1=MRD result available)")

Death_recorded flag (1=death recorded, 0=no death/unknown)
Had_Relapse_Specimen flag (1=relapse specimen available)
Day29_MRD_tested flag (1=MRD result available)


In [19]:
print("=" * 70)
print("STEP 3: DATA TYPE COERCION & VALUE CLEANING")
print("="*70)

STEP 3: DATA TYPE COERCION & VALUE CLEANING


In [20]:
df_clean = df.copy()

for col in continuous_vars: 
    if col not in df_clean.columns:
        continue
    before = df_clean[col].dtype
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    #errors= coerce converts non-numeric strings to NaN rather than crashing
    after = df_clean[col].dtype
    n_coerced = df_clean[col].isna().sum() - df[col].isna().sum()
    print(f" {col}: {before} -> {after} | {n_coerced} values coerced to NaN")

 Age.at.Diagnosis.in.Years: int64 -> int64 | 0 values coerced to NaN
 WBC.at.Diagnosis: float64 -> float64 | 0 values coerced to NaN
 Percent.Blasts.Tumor.Sample.Diagnostic: float64 -> float64 | 0 values coerced to NaN
 Germline.with.Tumor.Contamination.%.Blasts: float64 -> float64 | 0 values coerced to NaN


In [21]:
#Age: Days vs Years redundancy

if all (c in df_clean.columns for c in
        ["Age.at.Diagnosis.in.Years", "Age.at.Diagnosis.in.Days"]):
    df_check = df_clean[
        ["Age.at.Diagnosis.in.Years", "Age.at.Diagnosis.in.Days"]
        ].dropna()
    corr = df_check["Age.at.Diagnosis.in.Years"].corr(df_check["Age.at.Diagnosis.in.Days"])
    print(f"\n Age Years vs Days correlation: r = {corr:.4f}")
    if corr > 0.99: 
        print("Confirmed redundant - dropping 'Age.at.Diagnosis.in.Days'")
        df_clean = df_clean.drop(columns=["Age.at.Diagnosis.in.Days"])
        continuous_vars = [c for c in continuous_vars if c != "Age.at.Diagnosis.in.Days"]

#EFS days vs EFS column check 
if efs_days_col in df_clean.columns and "EFS" in df_clean.columns: 
    df_efs_check = df_clean[[efs_days_col, "EFS"]].dropna()
    corr_efs = df_efs_check[efs_days_col].corr(df_efs_check["EFS"])
    print(f"\n '{efs_days_col}' vs 'EFS' correlation: r = {corr_efs:.4f}")
    if corr_efs > 0.99:
        print(f" Redundant - dropping '{efs_days_col}', keeping 'EFS'")
        df_cleean = df_clean.drop(columns=[efs_days_col], errors='ignore')

#strip whitespace from all string columns 
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols: 
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace({"nan": np.nan, "None": np.nan,
                                           "": np.nan, "NA": np.nan,
                                           "N/A": np.nan, "Unknown": np.nan})

#sex standardisations if datset has mixed coding (e.g. m/f, male/female etc.)
if "Sex" in df_clean.columns:
    sex_map = {
        "M": "Male", "F": "Female", "m": "Male", "f": "Female",
        "Male": "Male", "Female": "Female", 
        "1": "Male", "2":"Female", "0": "Female",
    }
    df_clean["Sex"] = df_clean["Sex"].map(sex_map)
    print(f"\n Sex value counts after standardisation:")
    print(f" {df_clean["Sex"].value_counts().to_dict()}")

#check units for WBC at diagnosis. Can be recorded as an absolute count or ×10³/μL"
if "WBC.at.Diagnosis" in df_clean.columns:
    wbc = df_clean["WBC.at.Diagnosis"].dropna()
    print(f"\n WBC.at.Diagnosis range: {wbc.min():.1f} - {wbc.max():.1f}")
    print(" Verify units: if max > 1000, likely raw count not ×10³/μL")


 Age Years vs Days correlation: r = 0.9985
Confirmed redundant - dropping 'Age.at.Diagnosis.in.Days'

 'Event-free.Survival.in.Days' vs 'EFS' correlation: r = 1.0000
 Redundant - dropping 'Event-free.Survival.in.Days', keeping 'EFS'

 Sex value counts after standardisation:
 {'Male': 989, 'Female': 346}

 WBC.at.Diagnosis range: 0.6 - 325000.0
 Verify units: if max > 1000, likely raw count not ×10³/μL


In [22]:
df_clean["WBC.at.Diagnosis"]/=1000
print(f"\n Cleaned shape: {df_clean.shape}")


 Cleaned shape: (1335, 31)


#### STEP 4: OUTLIERS

WBC at diagnosis has a well-known bimodal distribution (low-risk ALL vs. high-risk hyperleucocytosis). "Outliers" here may be the most clinically important patient and not to be silently removed without identifying.

In [ ]:
cont_present = [c for c in continuous_vars if c in df_clean.columns]
outlier_log = [] 

fig, axes = plt.subplots(1, len(cont_present),
                         figsize=(len(cont_present)*4, 5))

if len(cont_present) == 1:
    axes = [axes]

for col in cont_present:
    series = df_clean[col].dropna()
    print(f"{col}: n={len(series)}, unique={series.nunique()}, std={series.std():.4f}")

for i, col in enumerate(cont_present): 
    series = df_clean[col].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 3*IQR, Q3 + 3*IQR 

    n_out = ((series<lo) | (series>hi)).sum()
    outlier_log.append({"variable":col, "n":len(series), 
                        "median": round(series.median(), 2), 
                        "Q1": round(Q1, 2), "Q3": round(Q3, 2), 
                        "lower_fence": round(lo,2), "upper_fence": round(hi, 2), 
                        "n_outliers": n_out, "pct_outliers": round(n_out/len(series)*100,
    1)})

sns.violinplot(y=series, ax=axes[i], color="#4C8BBF", alpha=0.5, inner=None)
sns.stripplot(y=series, ax=axes[i], color="#1A3A5C", size=3, alpha=0.5, jitter=True)
axes[i].axhline(lo, color="red", ls='--', lw=1, label=f'3xIQR fence')
axes[i].set_title(f"{col}\nn_outliers={n_out}", fontsize=9)

print(f" {col}: median={series.median():.1f}, "
      f"range=[{series.min():.1f}, {series.max():.1f}], "
      f"outliers={n_out} ({n_out/len(series)*100:.1f}%)")

plt.suptitle("Continuous Variable Distribution with Outlier Fences",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{output_dir}/04_outlier_violin.png", bbox_inches="tight")
plt.close()
pd.DataFrame(outlier_log).to_csv(f"{output_dir}/04_outlier_report.csv", index=False)
print(f"\n Outlier plot -> {output_dir}/04_outlier_violin.png")
print(f" Outlier table -> {output_dir}/04_outlier_report.csv")

First three plots empty tells us that those three columns exist in cont_present but have no variation or are effectively constant/binary, so the violin plot fails silently.
Most likely causes:
    1. columns are binary or near-constant (e.g. 0/1 flags). violinplot         requires enough spread to estimate a kernel density, and silentl         skips/fails on flat data.
    2. columns contain mostly NaN after dropna() leaving too few points

In [24]:
#Log-transform WBC. At diagnosis, WBC in leukaemia is heavily right-skewed. 
#Log-transforming normalises this for Cox regression and makes hazard ratios per unit clinically meaningful

if "WBC.at.Diagnosis" in df_clean.columns:
    df_clean["WBC_log"] = np.log1p(df_clean["WBC.at.Diagnosis"])
    print("\n Created 'WBC_log' = log1p(WBC.at.Diagnosis)")
    print(" -> Use WBC_log in Cox regression (not raw WBC) for better model fit")


 Created 'WBC_log' = log1p(WBC.at.Diagnosis)
 -> Use WBC_log in Cox regression (not raw WBC) for better model fit


#### STEP 5 = ENCODE CATEGORICAL VARIABLES

Cox regression is a numeric model. We encode each categorical type with a strategy matched to its clinical meaning.

In [25]:
df_enc = df_clean.copy()

#Binary encodings
binary_maps = {
    "Sex": {"Female": 0, "Male": 1}, 
    "ETP.Status": {"Non-ETP": 0, "ETP": 1, "non-ETP": 0, "Unknown": np.nan},
    "In.TARGET.Cohort.(n=.264).RNASeq.and.WES": {"Yes": 1, "No": 0,
                                                 "yes": 1, "no": 0,
                                                 "1": 1, "0": 0},
}

for col, mapping in binary_maps.items():
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map(mapping)
        print(f" Binary '[col]': {mapping}")

#ordinal encodings
#CNS Status: CNS1 (no blasts) < CNS2 (few blasts) < CNS3 (overt disease)
#CNS3 is treated more aggressively; ordered encoding preserves monotonic risk relationship for Cox regression
ordinal_maps = {
    "CNS.Status": {"CNS1": 1, "CNS2": 2, "CNS3": 3, 
                   "CNS1E": 1, "CNS2E": 2},
    "End.of.Consolidation.MRD": {"Negative": 0, "Low Positive": 1, 
                                 "Positive": 2, "negative": 0, "positive": 2},
    "Day.29.morphologic.Response": {"M1": 1, "M2": 2, "M3": 3},
}

for col, mapping in ordinal_maps.items():
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map(mapping)
        print(f" Ordinal '{col}': {mapping}")

nominal_cols = [
    "Tumor.Specimen.Type",
    "Germline.Specimen.Type",
    "Event.Type", 
]
nominal_present = [c for c in nominal_cols if c in df_enc.columns]
if nominal_present: 
    df_enc = pd.get_dummies(df_enc, columns=nominal_present,
                            drop_first=True, dtype=int)
    print("f\n One-hot encoded: {nominal_present}")

print(f"\n Shape after encoding: {df_enc.shape}")

 Binary '[col]': {'Female': 0, 'Male': 1}
 Ordinal 'CNS.Status': {'CNS1': 1, 'CNS2': 2, 'CNS3': 3, 'CNS1E': 1, 'CNS2E': 2}
 Ordinal 'End.of.Consolidation.MRD': {'Negative': 0, 'Low Positive': 1, 'Positive': 2, 'negative': 0, 'positive': 2}
f
 One-hot encoded: {nominal_present}

 Shape after encoding: (1335, 38)


#### STEP 6: SURVIVAL ENDPOINT VALIDATION

TARGET data has three survival endpoints. Each must be validated independently. The event status columns (OS.status, EFS.status, DFS.status) should be 0/1 only. Time columns should be strictly positive. Also check for consistency i.e. cannot have DFS > OS (disease-free survival cannot be longer than overall survival)

In [26]:
surv_fig, surv_axes = plt.subplots(1, 3, figsize=(16, 4))

for ax_idx, (ep_name, ep) in enumerate(survival_endpoints.items()):
    t_col = ep['time']
    e_col = ep['event']

    if t_col not in df_enc.columns or e_col not in df_enc.columns:
        print(f" {ep_name}: columns not found - skipping")
        continue

    t = pd.to_numeric(df_enc[t_col], errors='coerce')
    e = pd.to_numeric(df_enc[e_col], errors='coerce')

    df_enc[t_col] = t
    df_enc[e_col] = e

    n_total = len(t)
    n_events = int(e.sum())
    n_censored = int((e == 0).sum())
    pct_event = e.mean()*100
    med_time = t.median()
    neg_times = int((t < 0).sum())
    zero_times = int((t == 0).sum())
    t_missing = int(t.isna().sum())
    e_missing = int(e.isna().sum())

    print(f" {ep_name}:")
    print(f" n={n_total} | Events={n_events} ({pct_event:.1f}%) | "
          f"Censored={n_censored} | Median time={med_time:.0f} days")
    print(f" Missing: time={t_missing}, event={e_missing}")
    if neg_times: 
        print(f" {neg_times} NEGATIVE time values!")
    if zero_times: 
        print(f" {zero_times} ZERO time values (add 0.5 day or investigate)")
    print()

#distribution plot
surv_axes[ax_idx].hist(t[e == 1].dropna(), bins=25,
                       color='#C0392B', alpha=0.7, label='Event')
surv_axes[ax_idx].hist(t[e == 0].dropna(), bins=25, 
                        color='#2980B9', alpha=0.7, label='Censored')
surv_axes[ax_idx].set_title(f"{ep_name}\n{n_events} events / {n_total}")
surv_axes[ax_idx].set_xlabel("Time (days)")
surv_axes[ax_idx].set_ylabel("Count")
surv_axes[ax_idx].legend(fontsize=8)

print("Cross-endpoint consistency checks:")
if all(c in df_enc.columns for c in ["OS", "EFS"]):
    violation = (df_enc["EFS"] > df_enc["OS"]).sum()
    print(f" EFS > OS: {violation} patients"
        + (" <- investigate!" if violation > 0 else "✓"))
if all(c in df_enc.columns for c in ["OS", "DFS"]):
    violation = (df_enc["DFS"] > df_enc["OS"]).sum()
    print(f" DFS > OS: {violation} patients"
        + (" <- investigate!" if violation > 0 else "✓"))

plt.suptitle("Survival Endpoint Distributions", fontsize=13)
plt.tight_layout()
plt.savefig(f"{output_dir}/06_survival_validation.png", bbox_inches='tight')
plt.close()
print(f"\n Survival validation plot -> {output_dir}/06_survival_validation.png")

 OS:
 n=1335 | Events=141 (10.6%) | Censored=1194 | Median time=2477 days
 Missing: time=0, event=0

 EFS:
 n=1335 | Events=220 (16.5%) | Censored=1115 | Median time=2383 days
 Missing: time=0, event=0

 DFS:
 n=1335 | Events=141 (13.9%) | Censored=874 | Median time=2443 days
 Missing: time=320, event=320
 3 ZERO time values (add 0.5 day or investigate)

Cross-endpoint consistency checks:
 EFS > OS: 1 patients <- investigate!
 DFS > OS: 0 patients✓

 Survival validation plot -> eda_outputs/06_survival_validation.png


#### STEP 7: UNIVARIATE EXPLORATORY ANALYSIS - DISTRIBUTIONS

In [27]:
cont_present = [c for c in continuous_vars + ["WBC_log"]
                if c in df_enc.columns]

fig, axes = plt.subplots(2, len(cont_present),
                         figsize=(len(cont_present) *4, 8))
normality_results = []

for i, col in enumerate(cont_present):
    data = df_enc[col].dropna()

#histogram 
    axes[0, i].hist(data, bins=30, color='#3498DB', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f"{col}\nn={len(data)}", fontsize = 8)
    axes[0, i].set_ylabel("Count")

#Q-Q plot row (tests normality visually - points on diagonal = normal)
#more informative than histograms for small samples
    stats.probplot(data, dist="norm", plot=axes[1, i])
    axes[1, i].set_title("Q-Q Plot", fontsize=8)

#Shapiro-Wilk test
#statistical test to determine if a dataset is normally distributed
    sample = data.sample(min(len(data), 5000), random_state=42)
    stat_sw, p_sw = shapiro(sample)
    normality_results.append({"variable": col, "n": len(data),
                              "skewness": round(data.skew(), 3),
                              "normal": p_sw > 0.05})
    axes[0, i].set_xlabel(
        f"Skew={data.skew():.2f}\nShapiro p={p_sw:.3f}", fontsize=7)

plt.suptitle("Continuous Variable Distributions (Histogram + Q-Q)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f"{output_dir}/07a_continuous_distributions.png",
            bbox_inches='tight')
plt.close()
pd.DataFrame(normality_results).to_csv(
    f"{output_dir}/07a_normality_tests.csv", index=False)
print(f" Continuous distributions -> {output_dir}/07a_continuous_distributions.png")


 Continuous distributions -> eda_outputs/07a_continuous_distributions.png


In [28]:
key_cat_cols = [
    "Sex", "CNS.Status", "ETP.STATUS",
    "Day.29.MRD", "End.of.Consolidation.MRD", 
    "Day.29.morphologic.Response", "Event.Type",
]
key_cat_present = [c for c in key_cat_cols if c in df_enc.columns]

if key_cat_present:
    ncols=4
    nrows=int(np.ceil(len(key_cat_present)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
    axes = np.array(axes).flatten()
    colors = ['#2980B9', '#C0392B', '#27AE60', '#F39C12', '#8E44AD']

    for i, col in enumerate(key_cat_present):
        counts = df_enc[col].value_counts(dropna=False)
        pcts = counts/counts.sum()*100
        bars = axes[i].bar(
            counts.index.astype(str), pcts.values,
            color=colors[:len(counts)], edgecolor='white', alpha=0.85)

        for bar, pct in zip(bars, pcts.values):
            axes[i].text(bar.get_x() + bar.get_width()/2,
                         bar.get_height() + 0.5, 
                         f'{pct:.1f}%', ha='center', fontsize=7)

        axes[i].set_title(col, fontsize=9)
        axes[i].set_ylabel("% of cohort")
        axes[i].tick_params(axis='x', rotation=35, labelsize=7)

        rare = [str(k) for k, v in zip(counts.index, pcts.values) if v < 5]
        if rare: 
            axes[i].set_xlabel(f" Rare (<5%): {rare}", fontsize=7, 
                               color='darkred')

    for j in range(i + 1,len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Key Clinical Variable Distributions - TARGET Cohort",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/07b_clinical_distributions.png", 
                bbox_inches='tight')
    plt.close()
    print(f" Clinical distributions -> {output_dir}/07b_clinical_distributions.png")

 Clinical distributions -> eda_outputs/07b_clinical_distributions.png


#### STEP 8: BIVARIATE EXPLORATORY ANALYSIS

In [29]:
cont_for_corr = [c for c in cont_present if c in df_enc.columns]
if len(cont_for_corr) >= 2:
    corr = df_enc[cont_for_corr].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', 
                center=0, vmin=-1, vmax=1, square=True, 
                linewidths=0.5, ax=ax, annot_kws={"size": 9})
    ax.set_title("Spearman Correlation - Continuous Clinical Variables\n"
                 "(|r| > 0.7 -> multicollinearity concern)", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/08a_correlation_heatmap.png", bbox_inches='tight')
    plt.close()
    print(f" Correlation heatmap -> {output_dir}/08a_correlation_heatmap.png")

 Correlation heatmap -> eda_outputs/08a_correlation_heatmap.png


In [30]:
if primary_event not in df_enc.columns:
    print(f" '{primary_event}' not found - skipping group comparisons")
else:
    e_col_data = pd.to_numeric(df_enc[primary_event], errors='coerce')
    group_results=[]

    for col in cont_for_corr:
        if col in [primary_time, "OS", "EFS", "DFS"]:
            continue
        g1 = df_enc.loc[e_col_data == 1, col].dropna()
        g0 = df_enc.loc[e_col_data == 0, col].dropna()
        if len(g1) < 3 or len(g0) < 3:
            continue
        stat, p = mannwhitneyu(g1, g0, alternative='two-sided')
        sig = "✓" if p < 0.05 else " "
        print(f" {sig} {col}: median(event)={g1.median():.2f}| "
              f"median(censored)={g0.median():.2f} | p={p:.4f}")
        group_results.append({"variable": col, "test": "Mann-Whitney U",
                              "median_event": round(g1.median(), 3), 
                              "median_censored": round(g0.median(), 3), 
                              "p_value": round(p, 4), "sig_p05": p < 0.05})

    encoded_cats = [c for c in df_enc.columns
                    if c not in cont_for_corr
                    and c not in [patient_id]
                    and c not in [ep['time'] for ep in survival_endpoints.values()]
                    and c not in [ep['event'] for ep in survival_endpoints.values()]
                    and df_enc[c].nunique() <= 10
                    and df_enc[c].dtype in ['int64', 'float64', 'int32']] 
    
    for col in encoded_cats:
        try:
            ct = pd.crosstab(df_enc[col], e_col_data)
            if ct.shape[0] < 2 or ct.shape[1] < 2:
                continue
            chi2_stat, p, dof, exp = chi2_contingency(ct)
            min_exp = exp.min()
            flag = " ⚠(low expected)" if min_exp < 5 else ""
            sig = "✓" if p < 0.05 else " "
            print(f"  {sig} {col}: chi2 p={p:.4f}{flag}")
            group_results.append({"variable": col, "test": "Chi-squared",
                                   "median_event": None, "median_censored": None,
                                   "p_value": round(p, 4), "sig_p05": p < 0.05})
        except Exception:
            pass

        group_df = pd.DataFrame(group_results).sort_values("p_value")
    group_df.to_csv(f"{output_dir}/08b_group_comparisons_OS.csv", index=False)
    print(f"\n  Group comparison table → {output_dir}/08b_group_comparisons_OS.csv")

   Age.at.Diagnosis.in.Years: median(event)=10.00| median(censored)=9.00 | p=0.3661
 ✓ WBC.at.Diagnosis: median(event)=0.12| median(censored)=0.07 | p=0.0017
   Percent.Blasts.Tumor.Sample.Diagnostic: median(event)=86.00| median(censored)=89.00 | p=0.3111
   Germline.with.Tumor.Contamination.%.Blasts: median(event)=46.00| median(censored)=50.00 | p=1.0000
 ✓ WBC_log: median(event)=0.11| median(censored)=0.07 | p=0.0017
    Sex: chi2 p=0.8320
  ✓ Death_recorded: chi2 p=0.0000
    Day29_MRD_tested: chi2 p=0.7679 ⚠(low expected)
    Tumor.Specimen.Type_bone marrow: chi2 p=0.8858
  ✓ Germline.Specimen.Type_Sorted Diagnostic Marrow: chi2 p=0.0004 ⚠(low expected)
    Germline.Specimen.Type_blood: chi2 p=0.2320
    Germline.Specimen.Type_bone marrow: chi2 p=0.3919 ⚠(low expected)
  ✓ Event.Type_Induction death: chi2 p=0.0030 ⚠(low expected)
  ✓ Event.Type_Induction failure: chi2 p=0.0000 ⚠(low expected)
  ✓ Event.Type_Relapse: chi2 p=0.0000
  ✓ Event.Type_Second Malignant Neoplasm: chi2 p=0.0

MRD vs OS clinical highlight. Boxplot of OS by MRD status gives immediate clinical validation that the data is correct.

In [31]:
if "Day.29.MRD" in df_enc.columns and "OS" in df_enc.columns:
    mrd_data = df_enc[["Day.29.MRD", "OS", "OS.status"]].dropna().copy()

    # Classify MRD into clinical groups
    def classify_mrd(val):
        if val < 0.1:      return "Negative\n(<0.1%)"
        elif val < 10.0:   return "Low Positive\n(0.1–10%)"
        else:              return "Positive\n(≥10%)"

    mrd_data["MRD_group"] = mrd_data["Day.29.MRD"].apply(classify_mrd)
    group_order = ["Negative\n(<0.1%)", "Low Positive\n(0.1–10%)", "Positive\n(≥10%)"]

    print(mrd_data["MRD_group"].value_counts())

    # Group and fill missing groups with empty list
    mrd_groups = (mrd_data.groupby("MRD_group")["OS"]
                           .apply(list)
                           .reindex(group_order)
                           .apply(lambda x: x if isinstance(x, list) else []))

    event_rates = (mrd_data.groupby("MRD_group")["OS.status"]
                            .mean()
                            .reindex(group_order)
                            .fillna(0) * 100)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Left: Boxplot of OS by MRD group ---
    data_to_plot = list(mrd_groups.values)
    tick_labels  = [f"{g}\nn={len(v)}" for g, v in zip(group_order, mrd_groups.values)]

    axes[0].boxplot(data_to_plot,
                    tick_labels=tick_labels,
                    patch_artist=True,
                    boxprops=dict(facecolor='#3498DB', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[0].set_ylabel("OS (days)")
    axes[0].set_title("Overall Survival by Day 29 MRD")

    # --- Right: Event rate bar chart ---
    colors = ['#2ECC71', '#F39C12', '#E74C3C']
    axes[1].bar(range(3), event_rates.values, color=colors, alpha=0.7)
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(tick_labels)
    axes[1].set_ylabel("OS event rate (%)")
    axes[1].set_title("Event Rate by Day 29 MRD")
    axes[1].set_ylim(0, 100)
    for i, v in enumerate(event_rates.values):
        axes[1].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')

    plt.suptitle("MRD Status vs Survival Outcomes — Day 29 (Paediatric T-ALL)", 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/08c_MRD_vs_OS.png", bbox_inches='tight')
    plt.close()
    print(f"\n  MRD vs OS plot → {output_dir}/08c_MRD_vs_OS.png")
else:
    print("Required columns not found:", 
          [c for c in ["Day.29.MRD", "OS", "OS.status"] if c not in df_enc.columns])

MRD_group
Negative\n(<0.1%)          870
Low Positive\n(0.1–10%)    358
Positive\n(≥10%)           100
Name: count, dtype: int64

  MRD vs OS plot → eda_outputs/08c_MRD_vs_OS.png


In [32]:
if "Age.at.Diagnosis.in.Years" in df_enc.columns:
    age = df_enc["Age.at.Diagnosis.in.Years"].dropna()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(age, bins=25, color='#8E44AD', edgecolor='white', alpha=0.8)
    ax.axvline(1,   color='red', ls='--', lw=1.5, label='Age 1')
    ax.axvline(10,  color='orange', ls='--', lw=1.5, label='Age 10 (peak ALL)')
    ax.axvline(18,  color='green', ls='--', lw=1.5, label='Age 18 (adult cutoff)')
    ax.set_xlabel("Age at Diagnosis (Years)")
    ax.set_ylabel("Number of Patients")
    ax.set_title(f"Age Distribution at Diagnosis\n"
                 f"Median={age.median():.1f}y | "
                 f"Range=[{age.min():.1f}, {age.max():.1f}]y")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/08d_age_distribution.png", bbox_inches='tight')
    plt.close()
    print(f"  Age distribution → {output_dir}/08d_age_distribution.png")

  Age distribution → eda_outputs/08d_age_distribution.png


#### STEP 9: EXPORT CLEAN DATASETS

In [33]:
df_enc.to_csv(f"{output_dir}/09_TARGET_clean_full.csv", index=False)
print(f"  Full cleaned dataset → {output_dir}/09_TARGET_clean_full.csv ({df_enc.shape})")

  Full cleaned dataset → eda_outputs/09_TARGET_clean_full.csv ((1335, 38))


In [34]:
exclude_from_model = diagnosis_cols + ["Cause.of.Death", "Missing.Information",
                                        "Relapse.Specimen.Type",
                                        "Germline.Specimen.Type"]
model_cols = [c for c in df_enc.columns if c not in exclude_from_model]
df_model = df_enc[model_cols].copy()
 
# Drop rows where ALL survival columns are missing
surv_cols = ["OS", "OS.status", "EFS", "EFS.status", "DFS", "DFS.status"]
surv_cols_present = [c for c in surv_cols if c in df_model.columns]
df_model = df_model.dropna(subset=surv_cols_present, how='all')
 
df_model.to_csv(f"{output_dir}/09_TARGET_model_ready.csv", index=False)
print(f"  Model-ready dataset  → {output_dir}/09_TARGET_model_ready.csv ({df_model.shape})")

  Model-ready dataset  → eda_outputs/09_TARGET_model_ready.csv ((1335, 30))


In [35]:
cont_present_final = [c for c in continuous_vars + ["WBC_log"]
                      if c in df_enc.columns]
summary_rows = []
for col in cont_present_final:
    d = df_enc[col].dropna()
    summary_rows.append({
        "variable": col, "type": "continuous",
        "n": len(d), "pct_missing": round(df_enc[col].isna().mean()*100, 1),
        "median": round(d.median(), 2),
        "IQR": f"{round(d.quantile(0.25), 2)}–{round(d.quantile(0.75), 2)}",
        "mean ± SD": f"{round(d.mean(),2)} ± {round(d.std(),2)}",
        "min": round(d.min(), 2), "max": round(d.max(), 2),
    })
 
key_cats_final = ["Sex", "ETP.STATUS", "CNS.Status",
                  "Day.29.MRD", "End.of.Consolidation.MRD",
                  "Day.29.morphologic.Response"]
for col in key_cats_final:
    if col not in df_enc.columns:
        continue
    d = df_enc[col].dropna()
    for val, cnt in d.value_counts().items():
        summary_rows.append({
            "variable": f"{col} = {val}", "type": "categorical",
            "n": cnt, "pct_missing": round(df_enc[col].isna().mean()*100, 1),
            "median": None, "IQR": f"{round(cnt/len(df_enc)*100,1)}%",
            "mean ± SD": None, "min": None, "max": None,
        })

pd.DataFrame(summary_rows).to_csv(
    f"{output_dir}/09_summary_statistics.csv", index=False)
print(f"  Summary statistics   → {output_dir}/09_summary_statistics.csv")

  Summary statistics   → eda_outputs/09_summary_statistics.csv


In [36]:
print("\n" + "=" * 70)
print("PREPROCESSING & EDA COMPLETE")
print("=" * 70)
print(f"  Patients in clean dataset : {df_enc.shape[0]}")
print(f"  Variables retained        : {df_enc.shape[1]}")
print(f"  OS event rate             : {pd.to_numeric(df_enc.get('OS.status'), errors='coerce').mean()*100:.1f}%")
print(f"  EFS event rate            : {pd.to_numeric(df_enc.get('EFS.status'), errors='coerce').mean()*100:.1f}%")
print()
print("  Files written to:", output_dir)
files = [
    "00_column_snapshot.csv", "01_sample_id_lookup.csv",
    "02_missing_audit.csv", "02_missing_data.png",
    "04_outlier_violin.png", "04_outlier_report.csv",
    "06_survival_validation.png",
    "07a_continuous_distributions.png", "07a_normality_tests.csv",
    "07b_clinical_distributions.png", "07c_diagnosis_breakdown.png",
    "08a_correlation_heatmap.png", "08b_group_comparisons_OS.csv",
    "08c_MRD_vs_OS.png", "08d_age_distribution.png",
    "09_TARGET_clean_full.csv", "09_TARGET_model_ready.csv",
    "09_summary_statistics.csv",
]
for f in files:
    print(f"  ├── {f}")
 
print()
print("  NEXT STEP → Univariate Cox Regression")
print("  Input file: 09_TARGET_model_ready.csv")
print("  Primary endpoint: OS (time=OS, event=OS.status)")
print("=" * 70)


PREPROCESSING & EDA COMPLETE
  Patients in clean dataset : 1335
  Variables retained        : 38
  OS event rate             : 10.6%
  EFS event rate            : 16.5%

  Files written to: eda_outputs
  ├── 00_column_snapshot.csv
  ├── 01_sample_id_lookup.csv
  ├── 02_missing_audit.csv
  ├── 02_missing_data.png
  ├── 04_outlier_violin.png
  ├── 04_outlier_report.csv
  ├── 06_survival_validation.png
  ├── 07a_continuous_distributions.png
  ├── 07a_normality_tests.csv
  ├── 07b_clinical_distributions.png
  ├── 07c_diagnosis_breakdown.png
  ├── 08a_correlation_heatmap.png
  ├── 08b_group_comparisons_OS.csv
  ├── 08c_MRD_vs_OS.png
  ├── 08d_age_distribution.png
  ├── 09_TARGET_clean_full.csv
  ├── 09_TARGET_model_ready.csv
  ├── 09_summary_statistics.csv

  NEXT STEP → Univariate Cox Regression
  Input file: 09_TARGET_model_ready.csv
  Primary endpoint: OS (time=OS, event=OS.status)
